# C12-classical-models — Practice p24 — Solution


Team A starts with soft voting: average the three calibrated positive-class probabilities, then threshold. Architectural diversity can reduce error, but correlated errors or bad calibration are the risk. Team B starts with bagging: independently fit controlled trees on bootstrap samples and majority vote to reduce variance; correlated or biased members limit the gain. Team C starts with a random forest: bootstrap rows and randomly subset features at each split to decorrelate trees; too few informative candidates can raise bias. Team D starts with AdaBoost: fit stumps sequentially to reweighted rows and take an alpha-weighted vote to attack underfit, while monitoring suspected noisy rows because repeated mistakes receive growing weight. Pin seed 20260804, data folds, estimator counts, and base controls, then compare every proposal with its relevant base under identical stratified CV. Hard voting uses labels; soft voting uses calibrated probabilities; bagging is independent resampling; forests add split-feature randomness; AdaBoost is sequential correction.


In [ ]:
import numpy as np

team_decisions_p24 = {
    "A": {"choice":"soft voting", "aggregation":"mean calibrated probability", "target":"diverse-model error", "risk":"correlated errors or miscalibration"},
    "B": {"choice":"bagging", "aggregation":"majority over bootstrap trees", "target":"variance", "risk":"correlated or biased trees"},
    "C": {"choice":"random forest", "aggregation":"majority over row-and-feature-randomized trees", "target":"decorrelation", "risk":"too few informative features"},
    "D": {"choice":"AdaBoost", "aggregation":"alpha-weighted sequential vote", "target":"stump bias", "risk":"noisy-row weight concentration"},
}
SEED_p24 = 20260804

def adaboost_update_p24(q, t, h):
    q=np.asarray(q,dtype=np.float64); t=np.asarray(t,dtype=np.float64); h=np.asarray(h,dtype=np.float64)
    error=float(q[t!=h].sum())
    alpha=float(0.5*np.log((1.0-error)/error))
    unnormalized=q*np.exp(-alpha*t*h)
    Z=float(unnormalized.sum())
    return {"error":error,"alpha":alpha,"unnormalized":unnormalized,"Z":Z,"updated":unnormalized/Z}

audit_update_p24 = adaboost_update_p24(
    np.full(4,0.25), np.array([-1.,-1.,1.,1.]), np.array([-1.,1.,1.,1.])
)


### Answer check


In [ ]:
assert {team: decision["choice"] for team,decision in team_decisions_p24.items()} == {
    "A":"soft voting","B":"bagging","C":"random forest","D":"AdaBoost"}
assert team_decisions_p24["B"]["target"] == "variance"
assert team_decisions_p24["C"]["target"] == "decorrelation"
assert "noisy" in team_decisions_p24["D"]["risk"]
assert SEED_p24 == 20260804
assert np.isclose(audit_update_p24["error"],0.25,atol=1e-12,rtol=1e-10)
assert np.isclose(audit_update_p24["alpha"],0.5*np.log(3.0),atol=1e-12,rtol=1e-10)
assert np.isclose(audit_update_p24["updated"].sum(),1.0,atol=1e-12,rtol=1e-10)
assert audit_update_p24["updated"][1] > audit_update_p24["updated"][0]
